# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list the available record sets, their `@id`s, and the fields within each record set. This will help you choose which entities to use for extraction and analysis.

In [ ]:
# List all record sets, their '@id', and their fields, all referenced by '@id'
record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found in the top-level metadata. Trying to enumerate records directly...')
else:
    for rs in record_sets:
        print(f"RecordSet name: {getattr(rs, 'name', '<no name>')}")
        print(f"  @id: {getattr(rs, '@id', '<no id>')}")
        print(f"  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - {getattr(field, '@id', '<no id>')} ({getattr(field, 'name', '<no name>')})")

# If record_sets is empty, we assume a single record set; get ids from dataset.records iterator
if not record_sets:
    print('Enumerating records (first 1) and inferring columns...')
    try:
        records = list(dataset.records())
        if records:
            print(f"Columns: {list(records[0].keys())}")
    except Exception as e:
        print(f"Could not enumerate records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

If the dataset defines no explicit record sets, we load all available records.

In [ ]:
import pprint
# Choose the '@id' for the record set, or None if there is only a default/main record set
record_sets = dataset.metadata.record_sets

if record_sets:
    record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in record_sets]
else:
    # If no record sets, set to None and extract all records
    record_set_ids = [None]

dataframes = {}
for record_set_id in record_set_ids:
    if record_set_id is not None:
        records = list(dataset.records(record_set=record_set_id))
    else:
        records = list(dataset.records())
    key = record_set_id or 'main'
    dataframes[key] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record_set_id: {record_set_id if record_set_id else 'main'}")
    print("Columns:", dataframes[key].columns.tolist())
    pprint.pprint(dataframes[key].head(3).to_dict())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's pick a numeric field (column), such as 'coverage_percent' or 'MW' (`Molecular Weight`), if present in the main record set, and demonstrate filtering, normalization, and optional grouping.

Replace the `<numeric_field>` and `<group_field>` placeholders with the appropriate column names from the DataFrame obtained above.


In [ ]:
# Identify a numeric field. Example: 'coverage_percent', 'MW', or 'peptide_count' in this proteomics dataset.
df_key = list(dataframes.keys())[0]
df = dataframes[df_key]

# Try to determine some likely numeric fields from the columns:
potential_numeric = [col for col in df.columns if df[col].dtype.kind in 'fi']
if not potential_numeric:
    # Try to cast some columns that might be numeric but imported as object
    import numpy as np
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            continue
    potential_numeric = [col for col in df.columns if df[col].dtype.kind in 'fi']

print(f"Potential numeric fields: {potential_numeric}")
if potential_numeric:
    numeric_field = potential_numeric[0]  # Select the first numeric field
    threshold = df[numeric_field].mean()  # Filter above mean as example

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical variable if present (e.g. 'description' or 'accession')
    potential_group = None
    for c in df.columns:
        cnt = df[c].nunique()
        if df[c].dtype == object and 1 < cnt < 20:
            potential_group = c
            break

    if potential_group:
        grouped_df = filtered_df.groupby(potential_group).mean(numeric_only=True)
        print(f"\nGrouped data by {potential_group}:")
        print(grouped_df.head())
else:
    print("No numeric fields found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we demonstrate a histogram for a numeric field and a scatter plot if enough numeric columns exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_key = list(dataframes.keys())[0]
df = dataframes[df_key]

# Histogram for a numeric field
potential_numeric = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if potential_numeric:
    field = potential_numeric[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[field].dropna(), kde=True)
    plt.title(f"Distribution of {field}")
    plt.xlabel(field)
    plt.show()

# Scatter plot if at least two numeric fields are available
if len(potential_numeric) > 1:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=df[potential_numeric[0]], y=df[potential_numeric[1]])
    plt.xlabel(potential_numeric[0])
    plt.ylabel(potential_numeric[1])
    plt.title(f"Scatterplot of {potential_numeric[0]} vs {potential_numeric[1]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides a detailed account of protein abundance and related properties in extracellular vesicles isolated from human mast cells.
- Using `mlcroissant` allowed us to easily parse schema-driven metadata and extract complex tabular records.
- After identifying numeric fields, we performed filtering, normalization, and explored their distributions.
- The dataset can be further analyzed for protein expression patterns, modifications, and to support downstream tasks such as biomarker discovery.